# CS3201T Machine Learning Mini Project
## Automated Dry Bean Classification Using Morphological Features & Ensemble Learning

**Student Name:** Ayush Vinod Jadhav  
**PRN Number:** RBT24CS056  
**Roll Number:** CS3152  
**Course:** CS3201T ? Machine Learning Mini Project  
**Department:** Department of Computer Engineering  
**Dataset:** UCI Machine Learning Repository ? Dry Bean Dataset  
**Task Type:** Multi-Class Supervised Classification (7 Varieties)


### Abstract & Project Motivation
Agricultural quality inspection and grain sorting require fast, non-destructive, and objective classification methods. In traditional agriculture, seed classification is conducted manually by visual inspection, which is subjective, labor-intensive, and error-prone.

This project implements an end-to-end Machine Learning pipeline to classify seven registered commercial dry bean varieties (*SEKER, BARBUNYA, BOMBAY, CALI, DERMASON, HOROZ, and SIRA*) based on 16 high-resolution morphological and shape features extracted from computer vision imagery. The pipeline incorporates data cleaning, domain-informed feature engineering, multiple classification models, 5-fold cross-validation, systematic hyperparameter optimization via RandomizedSearchCV, and a Soft Voting Ensemble.

In [ ]:
# Import required libraries
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn tools
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)
import joblib

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='viridis')
%matplotlib inline

## 1. Data Collection & Schema Understanding
The dataset is acquired from the UCI Machine Learning Repository:
- **Records:** 13,611 raw samples
- **Features:** 16 geometric shape measurements
- **Target:** 7 distinct dry bean classes (Class)

In [ ]:
DATA_PATH = os.path.join('..', 'data', 'dry_beans.csv')
df = pd.read_csv(DATA_PATH)

# Standardize column names
rename_dict = {'AspectRation': 'AspectRatio', 'roundness': 'Roundness'}
df.rename(columns=rename_dict, inplace=True)

print('Dataset Shape:', df.shape)
df.head()

In [ ]:
# Inspect data types, null values, and summary statistics
print('Missing Values per column:')
print(df.isnull().sum())
print('\nDuplicate Rows:', df.duplicated().sum())
print('\nClass Distribution:')
print(df['Class'].value_counts())
df.describe().T

## 2. Exploratory Data Analysis (EDA)
Understanding class balance, feature skewness, and inter-feature correlations.

In [ ]:
# Target Class Frequency Distribution
plt.figure(figsize=(9, 5))
ax = sns.countplot(data=df, y='Class', order=df['Class'].value_counts().index, palette='viridis')
plt.title('Class Distribution across 7 Dry Bean Varieties', fontsize=14, fontweight='bold')
plt.xlabel('Number of Specimens')
plt.ylabel('Bean Variety')
plt.show()

In [ ]:
# Correlation Matrix across features
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.show()

## 3. Data Preprocessing & Feature Engineering
1. **Deduplication:** Remove duplicate records to prevent identical samples in train and test splits.
2. **Feature Engineering:** Derive morphological ratios:
   - PerimeterToAreaRatio: Higher for elongated grains
   - AreaToConvexAreaRatio: Convexity metric
   - CompactnessTimesRoundness: Geometric regularity score
   - MajorMinorProduct: Size cross-metric
   - ShapeIndex: AspectRatio * Eccentricity

In [ ]:
# Clean duplicates
df_clean = df.drop_duplicates().reset_index(drop=True)
print(f'Rows before: {len(df)} | Rows after deduplication: {len(df_clean)}')

# Feature Engineering
df_clean['PerimeterToAreaRatio'] = df_clean['Perimeter'] / (df_clean['Area'] + 1e-8)
df_clean['AreaToConvexAreaRatio'] = df_clean['Area'] / (df_clean['ConvexArea'] + 1e-8)
df_clean['CompactnessTimesRoundness'] = df_clean['Compactness'] * df_clean['Roundness']
df_clean['MajorMinorProduct'] = df_clean['MajorAxisLength'] * df_clean['MinorAxisLength']
df_clean['ShapeIndex'] = df_clean['AspectRatio'] * df_clean['Eccentricity']

print('New feature count:', df_clean.shape[1] - 1)

## 4. Stratified Train-Test Split & Scaling (Leakage Prevention)
- Split: 80% Training, 20% Testing with stratification.
- Fit StandardScaler strictly on training set.

In [ ]:
X = df_clean.drop(columns=['Class'])
y = df_clean['Class']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.20, random_state=42, stratify=y_encoded
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train samples: {X_train_scaled.shape[0]}, Test samples: {X_test_scaled.shape[0]}')

## 5. Model Training & Baseline Comparison
Evaluating multiple diverse classification algorithms:

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1500, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=12, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=16, random_state=42, n_jobs=-1),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=7, weights='distance'),
    'Support Vector Machine': SVC(kernel='rbf', C=1.5, probability=True, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=120, max_depth=4, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    results[name] = {'Accuracy': acc, 'F1_Weighted': f1}
    print(f'{name:25s}: Accuracy = {acc:.4f} | F1 = {f1:.4f}')

## 6. Advanced Machine Learning Techniques
### A. Hyperparameter Optimization via RandomizedSearchCV
### B. Soft Voting Ensemble Combining Tuned Random Forest, SVM, and Gradient Boosting

In [ ]:
# Hyperparameter Tuning for Random Forest
param_dist = {
    'n_estimators': [100, 150, 200],
    'max_depth': [14, 18, 22, None],
    'min_samples_split': [2, 4],
    'min_samples_leaf': [1, 2],
}
search = RandomizedSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1), param_dist, n_iter=10, cv=5, random_state=42, n_jobs=-1)
search.fit(X_train_scaled, y_train)
best_rf = search.best_estimator_
print('Best RF Parameters:', search.best_params_)

# Voting Ensemble
voting_clf = VotingClassifier(
    estimators=[
        ('rf', best_rf),
        ('svm', models['Support Vector Machine']),
        ('gb', models['Gradient Boosting']),
    ],
    voting='soft',
    n_jobs=-1
)
voting_clf.fit(X_train_scaled, y_train)
voting_pred = voting_clf.predict(X_test_scaled)
voting_acc = accuracy_score(y_test, voting_pred)
print(f'Voting Ensemble Test Accuracy: {voting_acc:.4f}')

## 7. Model Evaluation & Error Analysis
Confusion Matrix, Precision/Recall Breakdown, and Feature Importance.

In [ ]:
# Confusion Matrix Display
fig, ax = plt.subplots(figsize=(9, 7))
cm = confusion_matrix(y_test, voting_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title('Confusion Matrix ? Voting Ensemble Classifier', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.show()

print('Classification Report:')
print(classification_report(y_test, voting_pred, target_names=le.classes_))

## 8. Real-Time Inference Demo
Validating prediction on single input sample.

In [ ]:
sample_input = X_test.iloc[0:1]
sample_scaled = scaler.transform(sample_input)
pred_idx = voting_clf.predict(sample_scaled)[0]
pred_label = le.inverse_transform([pred_idx])[0]
true_label = le.inverse_transform([y_test[0]])[0]

print(f'Actual Class:    {true_label}')
print(f'Predicted Class: {pred_label}')
print('Match:', pred_label == true_label)